In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
import pandas as pd
import numpy as np
import copy
import seaborn as sns
from scipy.stats import mannwhitneyu as mwu
from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from statsmodels.stats.multitest import fdrcorrection
from scipy.stats import wilcoxon
from scipy.optimize import curve_fit
from scipy.stats import fisher_exact
import os
from scipy.stats import combine_pvalues
from scipy.stats import spearmanr,pearsonr
from collections import Counter
import sys
from math import floor

sns.set_style("white")
sns.set(font_scale = 1.5)
sns.set_style("white")

hfont = {'fontname':'Arial'}
plt.rcParams["font.family"] = "Arial"

#Code borrowed heavily from here: https://stackoverflow.com/questions/62375034/find-non-overlapping-area-between-two-kde-plots
plt.rcParams.update(
    {"text.usetex": False}
)

palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

gobp = pd.read_csv("../DPSC_CNCC/GOBP_AccelEvol_Input.txt", sep= "\t")
d_BP = {}

for index, row in gobp.iterrows():
    d_BP[row["Term"]] = row["Genes"].split(";")

hpo = pd.read_csv("../DPSC_CNCC/HPO_AccelEvol_Input.txt", sep= "\t")
d_HPO = {}

for index, row in hpo.iterrows():
    d_HPO[row["Term"]] = row["Genes"].split(";")
    

In [ ]:
palette = {"Human accelerated":"#E31A1C", "Chimp accelerated":"#0058FF", "Not significant":"grey", "Human":"#E31A1C", "Chimp":"#0058FF"}

d_abrev = {"LiangSteinNeuron":"FC exc. neur.", "FetalChondrocytes":"F chond.", "SertoliMale":"FG sertoli", "preGC_IIaFemale":"FG preGC IIa",\
          "NeuralFemale":"FG neur.", "FetalGonadImmuneFemale":"FG immune", "VIP":"AC VIP inh. neur.", "LiangSteinProgenitor":"FC prog.",\
          "AdultHeartVentricularCardiomyocyte":"AH cardiomyo.", "AdultLoopOfHenle":"AK loop of henle", "FetalBrainNeurGlioblast_CB_VZ":"FCB glioblast",\
         "AdultProximalTubule":"AK prox. tub.", "FetalLeydigMale":"FG leydig", "SST":"AC SST inh neur.", "KosoyRoussosControlMicroglia":"AC microglia",\
         "FetalBrainFloorPlate":"FB fl. plate", "FetalArterialECs":"FH endoth.", "ASCT":"AC astro.", "FetalBrainCOP":"FB COP",\
         "AMY":"AA neur.", "PVALB":"AC PVALB inh neur.", "ITL23":"AC L2-3 IT neur.", "FetalBrainNeurCB_GNP_IPC_1":"FB inter. prog.", "FetalBrainNeurDAergic":"FB DA neur.",\
          "OGC":"AC Oligo.", "D1Pu":"AP D1 inh neur.", "FetalBrainNeurSerotonergic":"FB 5-HT neur.", "FetalBrainNeurDRG_2":"FS DRG neur.",\
          "FetalHeartPericytes":"FH peri.", "FetalHeartEndocardium":"FH endocard.", "FetalHeartCardiacFibroblasts":"FH fibro.", "FetalBrainNeurPurkinje_6":"FCB Purk. inh neur.",\
          "AdultHeartSmoothMuscle":"AH smooth musc.", "FetalBrainRoofPlate":"FB ro. plate"}


In [ ]:
def filter_nc(file):
    v = pd.read_csv(file, sep = "\t")
    #Genes that are directly adjacent to centromeres
    blacklist = ["SHCBP1", "TP53TG3C", "SPATA31A5", "SPATA31A6", "FCGR1B", "PPIAL4C", "ALG10B" "ALG10", "PROS1", "OR4C46", "OR4C12", "ZNF33B", \
                "POTEC", "ZNF716", "ZNF727", "TEKT4", "RPIA", "SPIN4", "ZXDA", "LOC441155", "PRIM2", "MTRNR2L1", "ZNF337", "ZNF254", "DEFB115", \
                "UQCRFS1", "ZNF254", "EMB", "ALG10", "ALG10B", "WSB1", "ZNF37A", "SPIDR", "HGSNAT", "POTED"]

    #Genes that are near or in IGV etc. areas
    blacklist2 = ["ZNF267", "VPREB1", "SEC22B", "TUBGCP5", "LGALS9", "FAM90A1", "PDE4DIP"]
    
    #Genes that seem a little iffy since they are embedded in regions with lots of the same gene
    iffy = ["CYP4F8", "CLEC6A", "VKORC1L1", "UGT2B7"]
    orth = pd.read_csv("Orthologs_AllenSestan_HumChpGorRheMarm.txt", sep = "\t")
    orth = orth[(orth["Chimpanzee homology type"] == "ortholog_one2one") & (orth["Gorilla homology type"] == "ortholog_one2one")]
    v = v[v["NearestGene"].isin(orth["Gene name"])]
    v = v[~v["NearestGene"].isin(blacklist + blacklist2)]
    
    olfs = []
    hlas = []
    ugt = []
    for i in v["NearestGene"]:
        if i.startswith("OR1") or i.startswith("OR2") or i.startswith("OR3") or i.startswith("OR4") or i.startswith("OR5") or i.startswith("OR6") or i.startswith("OR1") or i.startswith("OR7") or i.startswith("OR8") or i.startswith("OR9"):
            olfs.append(i)
        if i.startswith("HLA-"):
            hlas.append(i)
        if i.startswith("UGT"):
            ugt.append(i)
    
    #Remove olfactory receptors and HLA genes
    v = v[~v["NearestGene"].isin(olfs + hlas + ugt)]
    
    #Filter to only genes with sufficient nearby substituions
    v = v[(v["Species1 Sum Total_Vars"] > 100) | (v["Species2 Sum Total_Vars"] > 100)]

    #Recompute FDR after filtering
    v["FDR Difference"] = fdrcorrection(v["p-value Difference"])[1]
    v["FDR Difference Corr Tot"] = fdrcorrection(v["p-value Difference Corr Tot"])[1]
    v["FDR Difference Corr S"] = fdrcorrection(v["p-value Difference Corr S"])[1]
    
    return v.sort_values("p-value Difference Corr Tot")

In [ ]:
cgh = pd.read_csv("hg38.panTro6.gorGor6.ponAbe3.CGHsites.bed.gz", sep = "\t", header = None)
cgh = list(cgh[0] + ":" + cgh[2].astype(str))
hgc = pd.read_csv("hg38.panTro6.gorGor6.ponAbe3.HGCsites.bed.gz", sep = "\t", header = None)
hgc = list(hgc[0] + ":" + hgc[2].astype(str))

In [ ]:
z_hgc = zn[zn["Position"].isin(hgc)]
z_cgh = zn[zn["Position"].isin(cgh)]

In [ ]:
z_ngc = zn[~zn["Position"].isin(list(hgc) + list(cgh))]

In [ ]:
hgc_chimp = z_hgc[(z_hgc["SpecSup447"] > 250) & (z_hgc["PhyloP447"] > 6)].shape[0]
cgh_human = z_cgh[(z_cgh["SpecSup447"] > 250) & (z_cgh["PhyloP447"] > 6)].shape[0]
hgc_chimp

In [ ]:
hgc_chimp = z_ngc[(z_ngc["SpecSup447"] > 250) & (z_hgc["PhyloP447"] > 6) & (z_ngc["Derived"] == "H")].shape[0]
cgh_human = z_ngc[(z_ngc["SpecSup447"] > 250) & (z_cgh["PhyloP447"] > 6) & (z_ngc["Derived"] == "C")].shape[0]
hgc_chimp

In [ ]:
z_cghs = z_cgh[z_cgh["SpecSup447"] > 250]
z_hgcs = z_hgc[z_hgc["SpecSup447"] > 250]
z_nils = zn[~zn["Position"].isin(hgc + cgh)]

chimp = z_nils[(z_nils["SpecSup447"] > 250) & (z_nils["PhyloP447"] > 6) & (z_nils["Derived"] == "C")].shape[0]
human = z_nils[(z_nils["SpecSup447"] > 250) & (z_nils["PhyloP447"] > 6) & (z_nils["Derived"] == "H")].shape[0]

chimp_a = z_nils[(z_nils["SpecSup447"] > 250) & (z_nils["Derived"] == "C")].shape[0]
human_a = z_nils[(z_nils["SpecSup447"] > 250) & (z_nils["Derived"] == "H")].shape[0]

In [ ]:
dfpp = pd.DataFrame([[26086, 48512 - 26086], ["Human", "Chimp"]]).T
dfpp.columns = ["Number of highly conserved sites", "Derived lineage"]

fig, ax = plt.subplots(figsize=(2, 4), dpi=600)
sns.barplot(data = dfpp, x = "Derived lineage", y = "Number of highly conserved sites", hue = "Derived lineage", alpha = 0.85, palette = {"Human":palette["Human"], "Chimp":palette["Chimp"]})
#plt.axhline(149/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
#plt.axhline(128/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
#plt.axhline(23/2.1035714285714286, linestyle = "--", color = "black")
plt.ylabel("Number of highly conserved sites", size = 11)
plt.xlabel("", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
y, h = 26086 + 500, 0 
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 200,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.title("Removing ILS", size = 12)
plt.ylim(0, 28500)
plt.xlabel("Derived lineage")
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
pos = zn["Position"].str.split(":", expand=True)
zn["chrom"] = pos[0]
zn["pos"] = pos[1].astype(int)
zh = zn[zn["Derived"] == "H"]
zc = zn[zn["Derived"] == "C"]

zh["PhyloP447_nonneg"] = zh["PhyloP447"].clip(lower=0)
zc["PhyloP447_nonneg"] = zc["PhyloP447"].clip(lower=0)

# Read BED
gbgc = pd.read_csv(
    "human_gBGC_hg38.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end"]
)
zh["in_gBGC"] = positions_in_intervals(zh, gbgc)

gbgc = pd.read_csv(
    "chimp_gBGC_hg38.bed",
    sep="\t",
    header=None,
    names=["chrom", "start", "end"]
)

zc["in_gBGC"] = positions_in_intervals(zc, gbgc)

z_nws = pd.concat([zh, zc])
z_nws = z_nws[z_nws["in_gBGC"] == False]
z_nws

In [ ]:
chimp = z_nws[(z_nws["SpecSup447"] > 250) & (z_nws["PhyloP447"] > 6) & (z_nws["Derived"] == "C")].shape[0]
human = z_nws[(z_nws["SpecSup447"] > 250) & (z_nws["PhyloP447"] > 6) & (z_nws["Derived"] == "H")].shape[0]

chimp_a = z_nws[(z_nws["SpecSup447"] > 250) & (z_nws["Derived"] == "C")].shape[0]
human_a = z_nws[(z_nws["SpecSup447"] > 250) & (z_nws["Derived"] == "H")].shape[0]

In [ ]:
chimp

In [ ]:
human

In [ ]:
from scipy.stats import binomtest
binomtest(human, human + chimp, p = human_a/(chimp_a + human_a))

In [ ]:
dfpp = pd.DataFrame([[26619, 23404], ["Human", "Chimp"]]).T
dfpp.columns = ["Number of highly conserved sites", "Derived lineage"]

fig, ax = plt.subplots(figsize=(2, 4), dpi=600)
sns.barplot(data = dfpp, x = "Derived lineage", y = "Number of highly conserved sites", hue = "Derived lineage", alpha = 0.85, palette = {"Human":palette["Human"], "Chimp":palette["Chimp"]})
#plt.axhline(149/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.055, xmax = 0.4535)
#plt.axhline(128/(149 + 128)*(33), linestyle = "--", color = "black", xmin = 0.5 + 0.055, xmax = 0.5 + 0.4535)
#plt.axhline(23/2.1035714285714286, linestyle = "--", color = "black")
plt.ylabel("Number of highly conserved sites", size = 11)
plt.xlabel("", size = 11)
plt.xticks(size = 10)
plt.yticks(size = 10)
x1, x22 = 0, 1                 # bar positions
y, h = 26619 + 500, 0 
plt.plot([x1, x1, x22, x22],
         [y, y + h, y + h, y],
         lw=1.2, c='black')

plt.text((x1 + x22) * 0.5, y - 200,
         "***",
         ha='center', va='bottom', fontsize=14)
plt.title("Removing gBGC", size = 12)
plt.ylim(0, 26619 + 2700)
plt.xlabel("Derived lineage")
#plt.title("Purkinje cells, ATAC peaks near Tenm3", size = 15)

In [ ]:
za = pd.read_csv("HumChp_All_PhyloP.txt.gz", sep = "\t")

zm = za[za["Category"] == "Mis"]
zn = za[(za["Category"] == "NC") | (za["Category"] == "NC_Syn")]
z3u = za[(za["Category"] == "3UTR") | (za["Category"] == "3UTR_Syn")]
z5u = za[(za["Category"] == "5UTR") | (za["Category"] == "5UTR_Syn")]

In [ ]:
z3u = za[(za["Category"] == "3UTR") | (za["Category"] == "3UTR_Syn")]


In [ ]:
cgh = pd.read_csv("hg38.panTro6.gorGor6.ponAbe3.CGHsites.bed.gz", sep = "\t", header = None)
cgh = list(cgh[0] + ":" + cgh[2].astype(str))
hgc = pd.read_csv("hg38.panTro6.gorGor6.ponAbe3.HGCsites.bed.gz", sep = "\t", header = None)
hgc = list(hgc[0] + ":" + hgc[2].astype(str))

def compute_ils(z, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table5_NoHeader.csv"):
    
    z = z[(z["SpecSup447"] > 250) & (z["PhyloP447"] > 0)]
    if "Table5" not in file:
        z = z[z["KeptAfterFilt"] == "Y"]
    zh = z[(z["Derived"] == "H") & (z["SpecSup447"] > 250)]
    zc = z[(z["Derived"] == "C") & (z["SpecSup447"] > 250)]
    
    z_hgc = z[z["Position"].isin(hgc)]
    z_cgh = z[z["Position"].isin(cgh)]

    z_hgc["PhyloP447_nonneg"] = z_hgc["PhyloP447"].clip(lower=0)
    z_cgh["PhyloP447_nonneg"] = z_cgh["PhyloP447"].clip(lower=0)
    
    z_hgcp = (
        z_hgc.groupby("NearestGene")
             .agg(
                 PhyloP447_sum=("PhyloP447_nonneg", "sum"),
                 n=("PhyloP447_nonneg", "size")
             )
    )
    
    z_cghp = (
        z_cgh.groupby("NearestGene")
             .agg(
                 PhyloP447_sum=("PhyloP447_nonneg", "sum"),
                 n=("PhyloP447_nonneg", "size")
             )
    )
    
    z_hgcp.columns = ["HGC Sum NonNeg PhyloP", "HGC Num NonNeg PhyloP"]
    z_cghp.columns = ["CGH Sum NonNeg PhyloP", "CGH Num NonNeg PhyloP"]
    zj = pd.read_csv(file)
    if "Table3" in file:
        zj = zj[zj["UTR"] == "5' UTR"]
    if "Table1" not in file:
        zj = zj.set_index("Gene symbol").join(z_hgcp).join(z_cghp).fillna(0)
    else:
        zj = zj.set_index("Protein gene symbol").join(z_hgcp).join(z_cghp).fillna(0)
    total_diff = (
        zj["Human PhyloP-weighted sum"]
        - zj["Chimp PhyloP-weighted sum"]
    )
    
    ils_diff = (
        zj["CGH Sum NonNeg PhyloP"]
        - zj["HGC Sum NonNeg PhyloP"]
    )
    
    zj["Fraction explained by ILS"] = np.where(
        (total_diff * ils_diff > 0),  # same sign
        ils_diff / total_diff,
        0
    )
    
    zj = zj.sort_values("Fraction explained by ILS")
    return zj

In [ ]:
zj = compute_ils(z5u, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3.csv")

In [ ]:
zj.loc[
    zj["Acceleration"] == "Not accelerated",
    "Fraction explained by ILS"
] = np.nan
zj.to_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3_5UTR_WithILS.csv")

In [ ]:
zj = zj[zj["Acceleration"] != "Not accelerated"]

In [ ]:
zj[(zj["Acceleration"] == "Human-accelerated, FDR < 0.05")].sort_values("Fraction explained by ILS").shape

In [ ]:
import pandas as pd
import numpy as np

def positions_in_intervals(df, intervals):
    keep = np.zeros(len(df), dtype=bool)

    for chrom, idx in df.groupby("chrom").groups.items():
        iv = intervals[intervals["chrom"] == chrom]

        if len(iv) == 0:
            continue

        starts = np.sort(iv["start"].to_numpy())
        ends = np.sort(iv["end"].to_numpy())

        p = df.loc[idx, "pos"].to_numpy()

        # Number of intervals that have started minus number that have ended
        n_started = np.searchsorted(starts, p, side="left")
        n_ended = np.searchsorted(ends, p, side="left")

        keep[df.index.get_indexer(idx)] = (n_started - n_ended) > 0

    return keep

def compute_gbgc(z, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table1_WithILS_WithAccelDecel.csv"):

    z = z[(z["SpecSup447"] > 250) & (z["PhyloP447"] > 0)]
    if "Table5" not in file:
        z = z[z["KeptAfterFilt"] == "Y"]
    
    # Extract chromosome and position from chr:Pos
    pos = z["Position"].str.split(":", expand=True)
    z["chrom"] = pos[0]
    z["pos"] = pos[1].astype(int)
    zh = z[z["Derived"] == "H"]
    zc = z[z["Derived"] == "C"]

    zh["PhyloP447_nonneg"] = zh["PhyloP447"].clip(lower=0)
    zc["PhyloP447_nonneg"] = zc["PhyloP447"].clip(lower=0)
    
    # Read BED
    gbgc = pd.read_csv(
        "human_gBGC_hg38.bed",
        sep="\t",
        header=None,
        names=["chrom", "start", "end"]
    )
    zh["in_gBGC"] = positions_in_intervals(zh, gbgc)
    
    gbgc = pd.read_csv(
        "chimp_gBGC_hg38.bed",
        sep="\t",
        header=None,
        names=["chrom", "start", "end"]
    )
    
    zc["in_gBGC"] = positions_in_intervals(zc, gbgc)

    zc.loc[zc["MutCat"] != "WS", "in_gBGC"] = False
    zh.loc[zh["MutCat"] != "WS", "in_gBGC"] = False

    zh = zh[zh["in_gBGC"] == True]
    zc = zc[zc["in_gBGC"] == True]
    zhp = (
        zh.groupby("NearestGene")
             .agg(
                 PhyloP447_sum=("PhyloP447_nonneg", "sum"),
                 n=("PhyloP447_nonneg", "size")
             )
    )
    
    zcp = (
        zc.groupby("NearestGene")
             .agg(
                 PhyloP447_sum=("PhyloP447_nonneg", "sum"),
                 n=("PhyloP447_nonneg", "size")
             )
    )
    
    zhp.columns = ["Human gBGC Sum NonNeg PhyloP", "Human gBGC Num NonNeg PhyloP"]
    zcp.columns = ["Chimp gBGC Sum NonNeg PhyloP", "Chimp gBGC Num NonNeg PhyloP"]
    zj = pd.read_csv(file)

    if "Table1" not in file:
        zj = zj.set_index("Gene symbol").join(zhp).join(zcp).fillna(0)
    else:
        zj = zj.set_index("Protein gene symbol").join(zhp).join(zcp).fillna(0)
    total_diff = (
        zj["Human PhyloP-weighted sum"]
        - zj["Chimp PhyloP-weighted sum"]
    )
    
    ils_diff = (
        zj["Human gBGC Sum NonNeg PhyloP"]
        - zj["Chimp gBGC Sum NonNeg PhyloP"]
    )
    
    zj["Fraction explained by gBGC"] = np.where(
        (total_diff * ils_diff > 0),  # same sign
        ils_diff / total_diff,
        0
    )
    
    zj = zj.sort_values("Fraction explained by gBGC")
    return zj, zh, zc, zhp, zcp
zj, zh, zc, zhp, zcp = compute_gbgc(z3u, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3_3UTR_WithILS_WithAccelDecel.csv")
zj

In [ ]:
gbgc = pd.read_csv(
        "chimp_gBGC_hg38.bed",
        sep="\t",
        header=None,
        names=["chrom", "start", "end"]
    )
np.sum(gbgc["end"] - gbgc["start"])

In [ ]:
gbgc = pd.read_csv(
        "human_gBGC_hg38.bed",
        sep="\t",
        header=None,
        names=["chrom", "start", "end"]
    )
np.sum(gbgc["end"] - gbgc["start"])

In [ ]:
zj.loc[
    zj["Acceleration"] == "Not accelerated",
    "Fraction explained by ILS"
] = np.nan
zj["Fraction explained by gBGC"] = zj["Fraction explained by gBGC"].clip(lower=0, upper=1)
zj.to_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3_3UTR_WithILS_WithAccelDecel_WithgBGC.csv")
zc = zj.copy()

In [ ]:
zj = pd.read_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table5_NoHeader_WithILS_WithAccelDecel_WithgBGC.csv")
zj = zj[(zj["Acceleration"] == "Human-accelerated, FDR < 0.05") | (zj["Acceleration"] == "Chimpanzee-accelerated, FDR < 0.05")]
#zj = zj[(zj["Acceleration"] == "Nominally human-accelerated, p < 0.05") | (zj["Acceleration"] == "Nominally chimpanzee-accelerated, p < 0.05") | (zj["Acceleration"] == "Human-accelerated, FDR < 0.05") | (zj["Acceleration"] == "Chimpanzee-accelerated, FDR < 0.05")]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

human_labels = ["Human-accelerated, FDR < 0.05", "Nominally human-accelerated, p < 0.05"]
chimp_labels = ["Chimpanzee-accelerated, FDR < 0.05", "Nominally chimpanzee-accelerated, p < 0.05"]

# Define the four subsets
groups = {
    "All": zj,
    "gBGC < 0.2": zj[zj["Fraction explained by gBGC"] < 0.2],
    "ILS < 0.2": zj[zj["Fraction explained by ILS"] < 0.2],
    "Both < 0.2": zj[
        (zj["Fraction explained by gBGC"] < 0.2) &
        (zj["Fraction explained by ILS"] < 0.2)
    ]
}

# Count significant human- and chimp-accelerated genes in each subset
human_counts = [
    (df["Acceleration"].isin(human_labels)).sum()
    for df in groups.values()
]

chimp_counts = [
    (df["Acceleration"].isin(chimp_labels)).sum()
    for df in groups.values()
]

# Plot
x = np.arange(len(groups))
width = 0.36

fig, ax = plt.subplots(figsize=(8, 5))

bars_h = ax.bar(
    x - width / 2,
    human_counts,
    width,
    color="#E31A1C",
    label="Human", alpha = 0.85
)

bars_c = ax.bar(
    x + width / 2,
    chimp_counts,
    width,
    color="#0058FF",
    label="Chimp", alpha = 0.85
)

ax.set_xticks(x)
ax.set_xticklabels(groups.keys())
ax.set_ylabel("Number of nominally accel. proteins")
ax.legend(frameon=False, bbox_to_anchor = (0.81, 0.896))

# Optional: put counts above bars
ax.bar_label(bars_h, padding=3)
ax.bar_label(bars_c, padding=3)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.title("Nominal HAPs and CAPs")

plt.tight_layout()
plt.show()

In [ ]:
z_gor = pd.read_csv("GorDer_AllSubs_Input_Final.bed.gz", sep = "\t", header = None)


In [ ]:
z_gor.columns = ["Chrom", "Pos", "Hum", "Gor", "Category", "PhyloP447", "SpecSup447", "NearestGene", "NearestDist"]
z_gor["Position"] = z_gor["Chrom"] + ":" + z_gor["Pos"].astype(str)
z_gor = z_gor.drop(["Chrom", "Pos"], axis = 1)


In [ ]:
z_gor = z_gor[(z_gor["PhyloP447"] != ".") & (z_gor["SpecSup447"] != ".")]
z_gor["PhyloP447"] = z_gor["PhyloP447"].astype(float)
z_gor["SpecSup447"] = z_gor["SpecSup447"].astype(float)
z_gor = z_gor[(z_gor["PhyloP447"] > 0) & (z_gor["SpecSup447"] > 250)]
#z_gor["NearestGene"] = z_gor["NearestGeneGene"]
#z_gor = z_gor.drop(["NearestGeneGene"], axis = 1)

In [ ]:
z_gor_m = z_gor[z_gor["Category"] == "Mis"]
z_gor_n = z_gor[z_gor["Category"] == "NC"]
z_gor_3u = z_gor[z_gor["Category"] == "3UTR"]
z_gor_5u = z_gor[z_gor["Category"] == "5UTR"]




In [ ]:
def do_gor(z, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table5_NoHeader_WithILS_WithgBGC.csv"):
    z_gor_mp = (
        z.groupby(["NearestGene"])["PhyloP447"]
           .sum()
    )
    z_gor_mp = pd.DataFrame(z_gor_mp)
    z_gor_mp.columns = ["PhyloP Sum GorDer"]
    if "Table1" in file:
        zj = pd.read_csv(file).set_index("Protein gene symbol")
    else:
        zj = pd.read_csv(file).set_index("Gene symbol")
    zj = zj.join(z_gor_mp).fillna(0)
    zj["HumGorDif"] = zj["Human PhyloP-weighted sum"] - zj["PhyloP Sum GorDer"]
    zj["ChpGorDif"] = zj["Chimp PhyloP-weighted sum"] - zj["PhyloP Sum GorDer"]
    
    H = "Human PhyloP-weighted sum"
    C = "Chimp PhyloP-weighted sum"
    G = "PhyloP Sum GorDer"
    
    # Expected H/C total if the two lineages were pooled
    hc_total = (zj[H].sum() + zj[C].sum()) / 2
    
    # Genome-wide excess on the gorilla branch
    gorilla_factor = zj[G].sum() / hc_total
    
    print("Gorilla scaling factor:", gorilla_factor)
    
    # Put gorilla on the H/C scale
    zj["Gorilla PhyloP-weighted sum corrected"] = (
        zj[G] / gorilla_factor
    )
    
    # Outgroup contrasts
    zj["Human vs Gorilla corrected"] = (
        zj[H] - zj["Gorilla PhyloP-weighted sum corrected"]
    )
    
    zj["Chimp vs Gorilla corrected"] = (
        zj[C] - zj["Gorilla PhyloP-weighted sum corrected"]
    )
    print(zj[[H, C, G, "Gorilla PhyloP-weighted sum corrected"]].sum())

    print(
        zj[[
            "Human vs Gorilla corrected",
            "Chimp vs Gorilla corrected"
        ]].describe()
    )
    H = "Human PhyloP-weighted sum"
    C = "Chimp PhyloP-weighted sum"
    G = "Gorilla PhyloP-weighted sum corrected"
    HC = "Difference in PhyloP-weighted sum"
    
    # Pairwise differences, if useful to retain
    zj["HumGorDif_corrected"] = zj[H] - zj[G]
    zj["ChpGorDif_corrected"] = zj[C] - zj[G]
    
    # Initialize
    zj["Fraction acceleration"] = np.nan
    zj["Fraction deceleration"] = np.nan
    zj["Acceleration lineage"] = np.nan
    zj["Deceleration lineage"] = np.nan
    
    # Human > chimp
    m_human = zj[HC] > 0
    
    zj.loc[m_human, "Fraction acceleration"] = (
        (zj.loc[m_human, H] - zj.loc[m_human, G])
        / zj.loc[m_human, HC]
    )
    
    zj.loc[m_human, "Fraction deceleration"] = (
        (zj.loc[m_human, G] - zj.loc[m_human, C])
        / zj.loc[m_human, HC]
    )
    
    zj.loc[m_human, "Acceleration lineage"] = "Human"
    zj.loc[m_human, "Deceleration lineage"] = "Chimp"
    
    
    # Chimp > human
    m_chimp = zj[HC] < 0
    
    # C - H is positive here
    chimp_minus_human = -zj.loc[m_chimp, HC]
    
    zj.loc[m_chimp, "Fraction acceleration"] = (
        (zj.loc[m_chimp, C] - zj.loc[m_chimp, G])
        / chimp_minus_human
    )
    
    zj.loc[m_chimp, "Fraction deceleration"] = (
        (zj.loc[m_chimp, G] - zj.loc[m_chimp, H])
        / chimp_minus_human
    )
    zj["Fraction acceleration clipped"] = (
        zj["Fraction acceleration"].clip(0, 1)
    )
    
    zj["Fraction deceleration clipped"] = (
        zj["Fraction deceleration"].clip(0, 1)
    )
    return zj

In [ ]:
zj = pd.read_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3_5UTR_WithILS_WithAccelDecel.csv")
np.mean(zj[zj["Acceleration"] == "Human-accelerated, FDR < 0.05"]["Fraction acceleration"])

In [ ]:
52/32

In [ ]:
21/13

In [ ]:
binomtest(52, 32 + 52)

In [ ]:
zj = do_gor(z_gor_5u, file = "Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table3_5UTR_WithILS.csv")

zc = zj.copy()

tp = {"Human accelerated":list(zc[(zc["Acceleration"] == "Human-accelerated, FDR < 0.05") | (zc["Acceleration"] == "Nominally human-accelerated, p < 0.05")]["Fraction acceleration clipped"]), \
     "Chimp accelerated":list(zc[(zc["Acceleration"] == "Chimpanzee-accelerated, FDR < 0.05") | (zc["Acceleration"] == "Nominally chimpanzee-accelerated, p < 0.05")]["Fraction acceleration clipped"])}
fig, ax = plt.subplots(figsize = (6, 4), dpi = 450)
sns.histplot(tp, palette = palette, bins = 50)
plt.xlabel("Fraction attributed to acceleration")
plt.ylabel("Number of genes")
plt.title("Accel. vs. decel. for nominal 5' HAUs and CAUs", size = 15)

In [ ]:
tp = {"Human accelerated":list(zc[(zc["Acceleration"] == "Human-accelerated, FDR < 0.05")]["Fraction acceleration clipped"]), \
     "Chimp accelerated":list(zc[(zc["Acceleration"] == "Chimpanzee-accelerated, FDR < 0.05")]["Fraction acceleration clipped"])}
fig, ax = plt.subplots(figsize = (6, 4), dpi = 450)
sns.histplot(tp, palette = palette, bins = 1)
plt.xlabel("Fraction attributed to acceleration")
plt.ylabel("Number of genes")
plt.title("Accel. vs. decel. for 5' HAUs and CAUs", size = 15)
plt.xlim(-0.02, 1.02)

In [ ]:
from scipy.stats import binomtest
v = pd.read_csv("Table_S5_ForBinom.csv")
x = []

for index, row in v.iterrows():
    x.append(binomtest(row["Number of human-derived substitutions"], row["Number of human-derived substitutions"] + row["Number of chimp-derived substitutions"]).pvalue)
v["Binomial p-value"] = x
v.to_csv("Table_S5_Binom.csv")

In [ ]:
#Making the weight-driven vs. count-driven calls for table S8
from scipy.stats import binomtest
ts8 = pd.read_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table8_New.csv").set_index("Gene").drop(["Maximum signed LFSR across cell types", "Minimum signed LFSR across cell types"], axis = 1) 
ts8 = ts8[(ts8["Human-acceleration for CA"] == "Human-accelerated CA in at least one cell type") | (ts8["Chimp-acceleration for CA"] == "Chimpanzee-accelerated CA in at least one cell type")]
#ts8 = ts8.drop(["Human-acceleration for CA", "Chimp-acceleration for CA"], axis = 1)

d_h = {}
dc_h = {}
d_c = {}
dc_c = {}
for i in ts8.index:
    d_h[i] = []
    dc_h[i] = []
    d_c[i] = []
    dc_c[i] = []
for i in os.listdir("ML_Results_Download"):
    ct = i.replace("_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results.txt", "")
    if "_AbsLogfc_NC_NoFiltWGS_PhyloP-100_SpecSup0_YCM_NCV_Per90_PerGene_Results" in i:
        v = pd.read_csv("ML_Results_Download/" + i, sep = "\t")
        v = v[v["NearestGene"].isin(ts8.index)]
        for index, row in v.iterrows():
            if ts8.loc[row["NearestGene"]][d_abrev[ct]] != '#NAME?':
                if row["p-value Difference Corr S"] < 0.05 and np.sign(row["Z-score Difference Corr S"]) == np.sign(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) and np.abs(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) > -np.log10(0.05):
                    if np.sign(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) == -1:
                        d_c[row["NearestGene"]].append(d_abrev[ct])
                    else:
                        d_h[row["NearestGene"]].append(d_abrev[ct])
                if binomtest(int(row["Species1 Sum Total_Vars"]), int(row["Species1 Sum Total_Vars"] + row["Species2 Sum Total_Vars"])).pvalue < 0.05 and np.sign(row["Z-score Difference Corr S"]) == np.sign(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) and np.abs(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) > -np.log10(0.05):
                    if np.sign(float(ts8.loc[row["NearestGene"]][d_abrev[ct]])) == -1:
                        dc_c[row["NearestGene"]].append(d_abrev[ct])
                    else:
                        dc_h[row["NearestGene"]].append(d_abrev[ct])

c = []
di = []
c_c = []
di_c = []
for i in ts8.index:
    c.append(";".join(dc_h[i]))
    di.append(";".join(d_h[i]))
    c_c.append(";".join(dc_c[i]))
    di_c.append(";".join(d_c[i]))
ts8["Human weight-driven"] = di
ts8["Human count-driven"] = c
ts8["Chimp weight-driven"] = di_c
ts8["Chimp count-driven"] = c_c
ts8.to_csv("Table_S8_Bleh.csv")

In [ ]:
ts82 = pd.read_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table8_New.csv").set_index("Gene")
ts82.join(ts8[["Human weight-driven", "Human count-driven", "Chimp weight-driven", "Chimp count-driven"]]).to_csv("Drafts/ToSubmit/Supplemental_Tables/Supplemental_Table8_New_WithDriven.csv")

In [ ]:
ts8[ts8["Human-acceleration for CA"] == "Human-accelerated CA in at least one cell type"].dropna(subset = "Human weight-driven").shape

In [ ]:
ts8[ts8["Human-acceleration for CA"] == "Human-accelerated CA in at least one cell type"].dropna(subset = "Human count-driven").shape

In [ ]:
ts8 = ts8.replace("", np.nan)

In [ ]:
z = pd.read_csv("LiangSteinNeuron_AllSitesToDownload.txt.gz", sep = "\t")


In [ ]:
zh = z[z["Derived"] == "H"]
zc = z[z["Derived"] == "C"]

In [ ]:
sns.histplot({"Human-derived":np.abs(zh["logfc"]), "Chimp-Derived":np.abs(zc["logfc"])})

In [ ]:
zhp = zh[(zh["SpecSup447"] > 250) & (zh["PhyloP447"] > 1)]

In [ ]:
zcp = zc[(zc["SpecSup447"] > 250) & (zc["PhyloP447"] > 1)]
zcp

In [ ]:
out = []
zhpp = zh[(zh["SpecSup447"] > 250) & (zh["PhyloP447"] < 0)]
#out.append(["< 0", zhpp[zhpp["logfc"] < -0.25].shape[0], zhpp[zhpp["logfc"] > 0.25].shape[0], "Human-derived"])
zcpp = zc[(zc["SpecSup447"] > 250) & (zc["PhyloP447"] < 0)]
#out.append(["< 0", zcpp[zcpp["logfc"] < -0.25].shape[0], zcpp[zcpp["logfc"] > 0.25].shape[0], "Chimp-derived"])
for b in [(0, 1), (1, 2), (2, 3), (3, 4), (4, 5), (5, 6), (6, 12)]:
    zhpp = zh[(zh["SpecSup447"] > 250) & (zh["PhyloP447"] > b[0]) & (zh["PhyloP447"] < b[1])]
    out.append([str(b[0]) + "-" + str(b[1]), zhpp[zhpp["logfc"] < -0.5].shape[0], zhpp[zhpp["logfc"] > 0.5].shape[0], "Human-derived"])
    zcpp = zc[(zc["SpecSup447"] > 250) & (zc["PhyloP447"] > b[0]) & (zc["PhyloP447"] < b[1])]
    out.append([str(b[0]) + "-" + str(b[1]), zcpp[zcpp["logfc"] < -0.5].shape[0], zcpp[zcpp["logfc"] > 0.5].shape[0], "Chimp-derived"])
df = pd.DataFrame(out)
df.columns = ["PhyloP bin", "Number increasing CA", "Number decreasing CA", "Derived"]
df["l2fc"] = np.log2(df["Number decreasing CA"]/df["Number increasing CA"])
df["PhyloP bin"] = df["PhyloP bin"].replace("6-12", "> 6")

In [ ]:
fig, ax = plt.subplots(figsize = (6, 4), dpi = 450)
sns.barplot(data = df[df["Derived"] == "Chimp-derived"], x = "PhyloP bin", y = "l2fc", color = "magenta")
plt.title("Bias toward decreasing CA in conserved sites\nfor chimp-derived substitutions", size = 15)
plt.xlabel("PhyloP bin", size = 13)
plt.ylabel("Log$_{2}$(decreasing CA/increasing CA)", size = 13)
plt.xticks(size = 12)
plt.yticks(size = 12)